## **집계연산**

### **PySpark 설정**

In [4]:
# [+] pyspark 불러오기
from pyspark import SparkConf, SparkContext

In [6]:
# [+] SparkConf, SparkContext 객체 생성하기
conf = SparkConf().setAppName("RDD Transformations and Actions").setMaster("local[*]")
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 04:19:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
sc = SparkContext(conf=conf)

### **reduce( ) 메서드**

In [ ]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x, y: x + y)

15

In [ ]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5], 2)
rdd.collect()

[1, 2, 3, 4, 5]

In [ ]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
glomed_rdd = rdd.glom()
glomed_rdd.collect()

[[1, 2], [3, 4, 5]]

In [ ]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
reduced_rdd = glomed_rdd.map(lambda x: reduce(lambda a, b: a + b, x))

15

In [ ]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5])

57

In [ ]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 2).reduce(lambda x,y: (x * 2) + y)

33

### **10000 개 정수 난수의 총합 계산하기**

In [ ]:
!pip install numpy

In [ ]:
# [+] numpy 라이브러리 불러오기
...

In [ ]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = ...
x[:100]

array([74, 10, 72, 30, 58, 10, 43, 68, 22, 47,  4, 22, 44,  0, 86, 13, 39,
       80, 68, 92, 89, 79, 35, 18, 18, 58, 70, 16, 44, 30, 84, 33, 16, 63,
       96,  5, 49, 67, 15, 83, 83,  3, 69, 97,  5, 15, 57, 57, 56, 17, 13,
       41, 88, 32, 92,  2,  9, 19, 64, 69, 81, 75, 88, 70, 67, 75, 57, 34,
       85,  6, 11, 55, 88, 89,  7, 74, 23, 43, 92,  3, 46, 46, 60, 91, 97,
       61, 78, 96, 53, 29, 40, 90, 62, 58, 54, 50, 84, 87, 58, 56])

In [ ]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = ...

In [ ]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")

In [ ]:
# [+] x+y 집계 연산
rdd.reduce(lambda x, y: x + y)

495388

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [ ]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4], 2).fold(1, lambda x, y: x + y)

12

In [ ]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4], 2).fold(1, lambda x, y: x + y)

13

In [ ]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0, lambda x, y: x * y)

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [7]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize([1, 2, 51,1,8,4,8,5,16,48,8]).groupBy(lambda x: x % 2).collect()

In [9]:
# [+] groupBy() 결과 출력하기
for k, v in res:
    print(f"Key: {k}, Values: {list(v)}")    

Key: 0, Values: [2, 8, 4, 8, 16, 48, 8]
Key: 1, Values: [1, 51, 1, 5]


### **aggregate( ) 메서드**

In [10]:
# reduce(), fold(): 입출력 RDD 형식이 같지 않으면 오류가 발생
sc.parallelize([1, 2, 3, 4])\
    .reduce(lambda x, y: (x + y, x * y)) 
    # single value -> pair 생성 시도할 경우 오류

TypeError: can only concatenate tuple (not "int") to tuple

In [ ]:
sc.parallelize([1, 2, 3, 4]).fold((0, 0), lambda x, y: (x[0] + y, x[1] + 1))

TypeError: unsupported operand type(s) for +: 'int' and 'tuple'

In [ ]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [ ]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0,0), seqOp, combOp)